In [ ]:

# P1-fix: os caminhos deste notebook são RELATIVOS a EXIST_VIDEOS/ (plano 11 §3).
# Se o kernel nascer noutro cwd, os asserts abaixo falham. Idempotente.
import os
from pathlib import Path

_TARGET = Path("/home/a67606/Thesis/EXIST_VIDEOS")
if Path.cwd() != _TARGET:
    assert _TARGET.is_dir(), f"diretório do projeto não encontrado: {_TARGET}"
    os.chdir(_TARGET)
assert Path("Dataset").is_dir(), f"cwd errado: {Path.cwd()} (sem Dataset/)"
assert Path("video_core.py").is_file(), "video_core.py não está no cwd"
print("cwd:", Path.cwd())

# ── Log em arquivo (independente de o Jupyter gravar outputs ou não) ────────
# O stdout é DUPLICADO: continua aparecendo na célula E vai para o arquivo.
# `_ORIG_STDOUT` guardado em globals() para re-executar esta célula não empilhar Tees.
import sys

_ORIG_STDOUT = globals().get("_ORIG_STDOUT", sys.stdout)


class _Tee:
    def __init__(self, path, term):
        self.f = open(path, "w", buffering=1, encoding="utf-8")
        self.term = term

    def write(self, s):
        self.term.write(s)
        self.f.write(s)
        return len(s)

    def flush(self):
        self.term.flush()
        self.f.flush()

    def isatty(self):
        return False


Path("analysis").mkdir(exist_ok=True)
LOG_PATH = "analysis/probe_report.txt"
sys.stdout = _Tee(LOG_PATH, _ORIG_STDOUT)
print("[log] stdout duplicado para", LOG_PATH,
      "\n[log] stderr NAO e capturado — se houver traceback, copiar da tela")



# Sondas de contribuição do ramo de vídeo — checkpoint `run_B2_bio_viewer_seed42_B15`

**O que este notebook é.** A execução dos cinco testes da §5 de
`improvements/15_VIDEO_INVESTIGATION_HANDOFF.md`. Sessão de **medição**, não de
implementação: nada é re-extraído, nada é re-treinado, nenhum encoder é trocado.

**O que este notebook NÃO é.** Evidência de que o vídeo ajuda. Regra explícita em
`video_core.py:1069`: a sonda é **gatilho** para gastar runs num braço treinado, nunca
evidência. Quem prova é um braço treinado.

---

## Decisão de design nº 1 — um forward, replays exatos

O `forward` do `EXISTVideoModel` (`video_core.py:1083-1164`) em `eval()` não tem
**nenhuma operação entre amostras**: `keep_* = None` (modality dropout desligado),
dropouts desligados, e é **proibido** por regra do projeto haver LayerNorm/BatchNorm
depois do `cat` (`video_core.py:1042-1046`). Logo cada um dos quatro `parts`
— `vid_feat`, `aud_feat`, `txt_feat`, `bio` — é função determinística **só** das
entradas da própria amostra.

Consequência: **permutar uma modalidade na entrada é idêntico a permutar o slot dela no
vetor concatenado.** Um único forward que cacheia o vetor de entrada da fusão permite
replay *exato* de:

- teste 1 (escada de permutação nas 4 modalidades),
- teste 2 (ruído pareado + 5 derangements + arms within-class),
- teste 5 (matriz de flips),

rodando só `fusion → heads` (milissegundos por arm). Isso não muda **o que** cada teste
mede — muda apenas como é computado, e derruba o custo de ~45 GPU-min para um forward.

⚠ Essa equivalência é uma afirmação, e afirmação sem verificação não vale nada neste
projeto. A célula **PORTÃO** abaixo aborta o notebook se o replay não reproduzir
(a) as métricas de `final_metrics.json` e (b) os dois números de permutação já medidos em
`test.ipynb` (Task B hard 0,4106 e 0,4316). Se o portão falhar, todo o resto é artefato de
replay e não pode ser lido.

## Decisão de design nº 2 — o replay fica na GPU, em bf16

`fusion → heads` é replayado dentro do **mesmo** `autocast(bf16)` do `trainer.predict`.
Replay em fp32 na CPU a partir de features bf16 introduziria divergência de dtype
exatamente na cabeça B, que é a hipótese sob teste (H3, fragilidade de fronteira). Só os
testes 4 (sklearn) e 5 (matriz de flips sobre logits já cacheados) são de facto CPU.

## Decisão de design nº 3 — captura num único ponto

Um `forward_hook` em `model.fusion[0]` captura a **entrada** da `Linear`, ou seja o vetor
concatenado como o modelo o fundiu, no dtype exato. Os quatro slots saem por fatias, com
os offsets lidos de `parts = [vid_feat, aud_feat, txt_feat, bio]`
(`video_core.py:1135-1146`): `0:768` vídeo · `768:1536` áudio · `1536:2560` texto ·
`2560:2688` bio. Quatro hooks separados poderiam dessincronizar; um não pode.

---

## Regra de leitura pré-registada (escrita ANTES de ver qualquer número)

As duas permutações já medidas deram Task B hard **−0,086** e **−0,065** — amplitude de
**0,021** sobre um efeito de −0,076. Portanto:

1. Todo arm é reportado como **média ± amplitude** sobre 5 derangements.
2. Uma diferença entre arms só é lida como diferença se **exceder a amplitude** dos
   próprios arms. Abaixo disso a leitura honesta é **inconclusivo a n=505**.
3. Este projeto já registrou um ganho de +0,020 que não replicou entre seeds. `n=1` não
   sustenta afirmação de tese.



## 0. Imports, checkpoint, split congelado


In [ ]:

import json, math, warnings
from collections import namedtuple

import numpy as np
import pandas as pd
import torch

import video_core as vc
from transformers import XLMRobertaTokenizer, TrainingArguments

warnings.filterwarnings("ignore", category=UserWarning)

RUN_DIR    = Path("run_B2_bio_viewer_seed42_B15")
CACHE_DIR  = Path("analysis/probe_cache_B15")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE = CACHE_DIR / "val_cache.pt"

N_DERANGEMENTS = 5
DERANGEMENT_SEEDS = list(range(N_DERANGEMENTS))     # 0..4; 0 e 1 = os de test.ipynb

# Offsets dos slots — lidos de video_core.py:1135-1146 (parts = [vid, aud, txt, bio]).
SLOTS = {
    "video": (0, vc.VIDEO_DIM),                                       # 0:768
    "audio": (vc.VIDEO_DIM, vc.VIDEO_DIM + vc.AUDIO_PROJ_DIM),        # 768:1536
    "texto": (vc.VIDEO_DIM + vc.AUDIO_PROJ_DIM,
              vc.VIDEO_DIM + vc.AUDIO_PROJ_DIM + vc.TEXT_DIM),        # 1536:2560
    "bio":   (vc.VIDEO_DIM + vc.AUDIO_PROJ_DIM + vc.TEXT_DIM,
              vc.fusion_input_dim(True)),                             # 2560:2688
}

METRIC_KEYS = [
    "task_a_icmnorm_hard", "task_a_icmsoftnorm_soft",
    "task_b_icmnorm_hard", "task_b_icmsoftnorm_soft",
    "task_c_icmnorm_hard", "task_c_icmsoftnorm_soft",
]

model, scaler = vc.load_run(RUN_DIR, init_seed=42)
with open(RUN_DIR / "meta.json") as f:
    META = json.load(f)
with open(RUN_DIR / "final_metrics.json") as f:
    FINAL = json.load(f)

print("\nmeta do checkpoint sondado:")
for k in ("condition", "arm", "restore_biases", "frames_version", "audio_version",
          "num_train_epochs", "fusion_in", "split_seed"):
    print(f"  {k:18s} {META.get(k)}")
assert META["fusion_in"] == vc.fusion_input_dim(True) == 2688, "fusion_in inesperado"
assert META["restore_biases"] is True, "este checkpoint NÃO tem ARCH-01 corrigido"

tokenizer = XLMRobertaTokenizer.from_pretrained(vc.XLMR_NAME)
full_df = pd.read_json("Dataset/training/EXIST2026_training.json").transpose()
train_df, val_df = vc.build_frozen_split(full_df)
val_ds = vc.EXISTVideoDataset(val_df, tokenizer, scaler,
                              vc.frames_dir("train"), vc.audio_dir("train"))
N_VAL = len(val_ds)
VAL_IDS = [str(i) for i in val_df["id_EXIST"].values]
print(f"\nval = {N_VAL} amostras (split congelado seed {vc.SPLIT_SEED})")
assert N_VAL == len(VAL_IDS) == 505, f"val tem {N_VAL}, esperado 505 (split congelado)"



## 1. Stage A — um forward sobre a val, cacheando tudo

Captura, num só `trainer.predict`:

- **entrada da fusão** `[505, 2688]` (os quatro slots, no dtype exato do modelo);
- **logits** das três cabeças e os **soft labels** (`label_ids`, em ordem de dataset);
- **diagnóstico do pool**: entropia do softmax por amostra, peso máximo, massa do top-10
  e o perfil somado por posição temporal.

O hook do pool fica em `model.video_pool.attn`, cuja **saída** é exatamente o tensor
`logits [B, 1568, 1]` de `video_core.py:965`. Não é recomputação: é o mesmo tensor.

Idempotente: se `analysis/probe_cache_B15/val_cache.pt` existe, o forward é saltado.


In [ ]:

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", ("Stage A exige GPU. Este notebook roda na VM, não na máquina "
                          "local de desenvolvimento.")

args = TrainingArguments(output_dir="/tmp/probe_vid", per_device_eval_batch_size=2,
                         bf16=True, remove_unused_columns=False, report_to="none",
                         label_names=["labels_a", "labels_b", "labels_c"])
trainer = vc.MultiTaskTrainer(model=model, args=args,
                              compute_metrics=vc.compute_metrics,
                              data_collator=vc.video_collate)
trainer.label_names = ["labels_a", "labels_b", "labels_c"]

# 16 frames / tubelet 2 = 8 posições temporais × 196 patches 14×14 = 1568 tokens (F6).
N_TEMPORAL = vc.NUM_FRAMES // 2
N_SPATIAL  = (vc.FRAME_H // 16) * (vc.FRAME_W // 16)
assert N_TEMPORAL * N_SPATIAL == 1568, (N_TEMPORAL, N_SPATIAL)


def _run_stage_a():
    buf_fusion, buf_pool = [], []

    def hook_fusion(_m, inp, _out):
        buf_fusion.append(inp[0].detach().float().cpu())

    def hook_pool(_m, _inp, out):
        # out = logits [B, 1568, 1] — o MESMO tensor de video_core.py:965.
        lg = out.detach().float()
        w = torch.softmax(lg, dim=1).squeeze(-1)                 # [B, 1568]
        ent = -(w.clamp_min(1e-12).log() * w).sum(dim=1)         # nats
        top10 = w.topk(10, dim=1).values.sum(dim=1)
        temporal = w.view(w.shape[0], N_TEMPORAL, N_SPATIAL).sum(dim=2)   # [B, 8]
        buf_pool.append((torch.stack([ent, w.max(dim=1).values, top10], dim=1).cpu(),
                         temporal.cpu()))

    h1 = model.fusion[0].register_forward_hook(hook_fusion)
    h2 = model.video_pool.attn.register_forward_hook(hook_pool)
    try:
        out = trainer.predict(val_ds)
    finally:
        h1.remove(); h2.remove()

    logits_a, logits_b, logits_c = vc._unpack_predictions(out.predictions)
    lab_a, lab_b, lab_c = out.label_ids

    cache = {
        "fusion_in": torch.cat(buf_fusion, dim=0),                       # [505, 2688]
        "logits_a": np.asarray(logits_a), "logits_b": np.asarray(logits_b),
        "logits_c": np.asarray(logits_c),
        "labels_a": np.asarray(lab_a), "labels_b": np.asarray(lab_b),
        "labels_c": np.asarray(lab_c),
        "pool_stats": torch.cat([b[0] for b in buf_pool], dim=0),        # [505, 3]
        "pool_temporal": torch.cat([b[1] for b in buf_pool], dim=0),     # [505, 8]
        "ids": VAL_IDS,
    }
    torch.save(cache, CACHE_FILE)
    return cache


if CACHE_FILE.exists():
    cache = torch.load(CACHE_FILE, weights_only=False)
    print(f"[cache] carregado de {CACHE_FILE} (forward saltado)")
else:
    cache = _run_stage_a()
    print(f"[cache] gravado em {CACHE_FILE}")

FUSION_IN = cache["fusion_in"]
LAB_A, LAB_B, LAB_C = cache["labels_a"], cache["labels_b"], cache["labels_c"]
assert FUSION_IN.shape == (N_VAL, 2688), FUSION_IN.shape
assert cache["pool_stats"].shape == (N_VAL, 3)
assert cache["ids"] == VAL_IDS, ("ordem do cache divergiu do val_df — cache antigo? "
                                 "apagar analysis/probe_cache_B15/ e re-rodar")
print("shapes OK:", {k: tuple(v.shape) for k, v in cache.items()
                     if hasattr(v, "shape")})



## 2. PORTÃO — o replay reproduz o forward completo?

O cache já contém os **logits do forward real** (`trainer.predict`). Isso permite separar
três causas de falha que um assert único confundiria:

| # | condição | o que significa se falhar |
|---|---|---|
| **A** | logits do replay == logits do forward | o replay está errado (dtype, batch, `train()`/`eval()`) — **aborta** |
| **B** | métricas do forward == `final_metrics.json` | esta sessão não reproduz a run de origem (ambiente/PyEvALL) — **avisa, não aborta** |
| **C** | replay+derangement == `test.ipynb` (B hard 0,4106 / 0,4316) | idem B, arrastado — **avisa** |

**Só a A é condição de validade deste notebook.** B e C comparam com números medidos
noutra sessão; se divergirem, os Δ *dentro* desta sessão continuam válidos (é Δ que
decide as hipóteses, não valor absoluto) — mas os absolutos não podem ser colados ao lado
dos do doc 15 sem a ressalva.

### Autocalibração do modo de replay

Quatro modos (`bs` ∈ {2, 64} × autocast bf16 on/off) são testados contra os logits
cacheados; ganha o que reproduz. Razões:

- **`bs=2`** casa com `per_device_eval_batch_size=2` do `predict`. Matmul bf16 reduz em
  tiles, então mudar o batch muda o último bit — e a cabeça B é fragilidade de fronteira
  por hipótese (H3), logo o último bit pode virar um `argmax`.
- **autocast on/off** porque não é dado que o `Trainer` da v5 autocaste no `predict`.

### `model.eval()` explícito

`nn.Module` nasce em `train()`. No caminho "cache carregado do disco" o modelo **nunca**
passa por `trainer.predict`, e o `Dropout(0.1)` da fusão (`video_core.py:1049`) estaria
ativo no replay. Essa era a falha nº 1 candidata.


In [ ]:

EvalPred = namedtuple("EvalPred", ["predictions", "label_ids"])

model.eval()
print("model.training =", model.training, "(tem de ser False)")
assert model.training is False

_FUSION_GPU = FUSION_IN.to(device)
REPLAY_BS, REPLAY_AUTOCAST = 2, True          # sobrescritos pela autocalibração abaixo


def replay(fusion_in_gpu, bs=None, use_autocast=None):
    """fusion -> heads. Modo (bs, autocast) vem da autocalibração desta célula."""
    bs = REPLAY_BS if bs is None else bs
    ac = REPLAY_AUTOCAST if use_autocast is None else use_autocast
    ctx = (torch.autocast("cuda", dtype=torch.bfloat16) if ac
           else torch.autocast("cuda", enabled=False))
    outs = ([], [], [])
    with torch.no_grad(), ctx:
        for i in range(0, fusion_in_gpu.shape[0], bs):
            fused = model.fusion(fusion_in_gpu[i:i + bs])
            outs[0].append(model.head_a(fused).squeeze(-1).float().cpu())
            outs[1].append(model.head_b(fused).float().cpu())
            outs[2].append(model.head_c(fused).float().cpu())
    return tuple(torch.cat(o).numpy() for o in outs)


def metrics_of(logits3):
    m = vc.compute_metrics(EvalPred(logits3, (LAB_A, LAB_B, LAB_C)))
    return {k: float(m[k]) for k in METRIC_KEYS}


def derangement(n, seed):
    """MESMA construção de ShuffledVideo em test.ipynb: permutação sem ponto fixo."""
    rs = np.random.RandomState(seed)
    while True:
        p = rs.permutation(n)
        if not (p == np.arange(n)).any():
            return p


def perturb(slot, index=None, replacement=None):
    """Cópia GPU de FUSION_IN com o slot substituído (permutação de linhas ou tensor)."""
    a, b = SLOTS[slot]
    x = FUSION_IN.clone()
    if index is not None:
        x[:, a:b] = FUSION_IN[np.asarray(index), a:b]
    if replacement is not None:
        x[:, a:b] = replacement.to(FUSION_IN.dtype)
    return x.to(device)


def delta_norm(slot, index=None, replacement=None):
    """||Dslot||2 médio por amostra — magnitude da perturbação realmente aplicada."""
    a, b = SLOTS[slot]
    base = FUSION_IN[:, a:b].float()
    new = base[np.asarray(index)] if index is not None else replacement.float()
    return float((new - base).norm(dim=1).mean())


# ================= autocalibração: qual modo reproduz o forward? =================
REF = (cache["logits_a"], cache["logits_b"], cache["logits_c"])
MODES = {"bs2+bf16": (2, True), "bs2+fp32": (2, False),
         "bs64+bf16": (64, True), "bs64+fp32": (64, False)}
cands = {}
print(f"\n{'modo':>10s}{'maxD logit':>14s}{'flips argmax B':>16s}{'flips A@0.5':>13s}")
for name, (bs, ac) in MODES.items():
    r = replay(_FUSION_GPU, bs=bs, use_autocast=ac)
    mx = max(float(np.abs(r[i] - REF[i]).max()) for i in range(3))
    fb = int((r[1].argmax(1) != REF[1].argmax(1)).sum())
    fa = int(((r[0] > 0) != (REF[0] > 0)).sum())
    cands[name] = (mx, fb, fa, bs, ac)
    print(f"{name:>10s}{mx:14.3e}{fb:16d}{fa:13d}")

best = min(cands, key=lambda k: (cands[k][1] + cands[k][2], cands[k][0]))
mx, fb, fa, REPLAY_BS, REPLAY_AUTOCAST = cands[best]
print(f"\nmodo escolhido: {best}  (bs={REPLAY_BS}, autocast_bf16={REPLAY_AUTOCAST})")

assert fb == 0 and fa == 0, (
    f"PORTAO A FALHOU: nenhum modo de replay reproduz as DECISOES do forward completo "
    f"(melhor: {best}, {fb} flips em B, {fa} em A, maxD logit {mx:.3e}). A equivalencia "
    f"fusion-in -> logits nao se sustenta nesta GPU/versao. Abandonar o replay: rodar os "
    f"testes 1/2/5 com forwards completos (custo ~45 GPU-min) e manter 3/4 sobre o cache.")
if mx > 1e-3:
    print(f"[AVISO] maxD logit = {mx:.3e} sem flip de decisao. Replay valido para metricas "
          f"hard; tratar as metricas SOFT com reserva.")
print(">>> PORTAO A OK: o replay reproduz as decisoes do forward completo.")

# ================= B: esta sessao reproduz final_metrics.json? =================
m_forward = metrics_of(REF)          # métricas pelo MESMO código, sobre os logits reais
m_normal = metrics_of(replay(_FUSION_GPU))

print(f"\n{'metrica':30s}{'forward':>10s}{'replay':>10s}{'final_metrics':>15s}"
      f"{'D repl-fwd':>12s}{'D fwd-final':>13s}")
gate_replay, gate_session = True, True
for k in METRIC_KEYS:
    ref = float(FINAL[f"eval_{k}"])
    d1, d2 = m_normal[k] - m_forward[k], m_forward[k] - ref
    gate_replay &= abs(d1) < 1e-3
    gate_session &= abs(d2) < 1e-3
    print(f"{k:30s}{m_forward[k]:10.4f}{m_normal[k]:10.4f}{ref:15.4f}"
          f"{d1:+12.5f}{d2:+13.5f}")

assert gate_replay, (
    "PORTAO A FALHOU no nivel das METRICAS: os logits do replay reproduzem as decisoes "
    "mas as metricas divergem >1e-3. Isso so pode vir de nao-determinismo dentro do "
    "PyEvALL (mesma entrada, saida diferente) — investigar antes de seguir.")

# ================= C: e os numeros de test.ipynb? =================
REF_SHUF_B_HARD = {0: 0.4106, 1: 0.4316}
print()
gate_shuf = True
for seed, ref in REF_SHUF_B_HARD.items():
    got = metrics_of(replay(perturb("video", index=derangement(N_VAL, seed))))["task_b_icmnorm_hard"]
    gate_shuf &= abs(got - ref) < 1e-3
    print(f"shuf{seed}  task_b_icmnorm_hard  replay={got:.4f}  test.ipynb={ref:.4f}  "
          f"D={got - ref:+.5f}")

SESSION_REPRODUCES = bool(gate_session and gate_shuf)
print()
if SESSION_REPRODUCES:
    print(">>> PORTOES B e C OK: esta sessao reproduz a run de origem. Valores ABSOLUTOS "
          "comparaveis com o doc 15.")
else:
    print("=" * 78)
    print("[AVISO ALTO] Esta sessao NAO reproduz os numeros da run de origem:")
    if not gate_session:
        print("  - metricas do forward != final_metrics.json (coluna 'D fwd-final' acima)")
    if not gate_shuf:
        print("  - permutacao != test.ipynb (0,4106 / 0,4316)")
    print("  O replay esta VALIDO (portao A passou), logo todos os DELTAS deste notebook")
    print("  sao internamente consistentes e as hipoteses H1/H2/H3/H5 continuam decidiveis")
    print("  — elas dependem de comparacao ENTRE arms da mesma sessao, nao de absolutos.")
    print("  MAS: nao colar os valores absolutos ao lado dos do doc 15 sem esta ressalva,")
    print("  e investigar a divergencia (versao de PyEvALL? scaler? split?) antes de")
    print("  qualquer numero deste notebook ir para a tese.")
    print("=" * 78)



## 3. Bloco inicial — as duas coisas que podem encerrar a investigação

Duas leituras que, se saírem de um certo jeito, mudam o **objeto** da investigação e
tornam os testes seguintes sem sentido. Por isso vêm primeiro.

### 3.1 A Task B prediz DIRECT alguma vez?

`eval_task_b_fmeasure_hard` é **NaN** no checkpoint e `eval_proxy_task_b_f1 = 0,6104` não
resolve a questão (macro-F1 de 3 classes com uma classe nunca predita teto em 0,667 — o
valor observado é compatível com as duas situações). A matriz de confusão resolve, de
graça.

**Por que importa:** o mecanismo que a H1 propõe é *"formato separa DIRECT de
JUDGEMENTAL"*. Se DIRECT nunca aparece no `argmax` da condição normal, o −0,076 é
movimento **só** entre NONE e JUDGEMENTAL, e o mecanismo declarado da H1 **não pode** ser
o que o produz. Isso restringe a investigação tanto quanto o teste 2.


In [ ]:

LABELS_B = list(vc.LABELS_2_2)
mask_b = LAB_B[:, 0] != vc.SENTINEL_BC
gold_b = LAB_B[mask_b].argmax(axis=1)
pred_b_normal = cache["logits_b"][mask_b].argmax(axis=1)

print(f"amostras com voto válido em B: {int(mask_b.sum())} de {N_VAL}\n")
print("ordem das classes:", LABELS_B, "\n")

cm = np.zeros((3, 3), dtype=int)
for g, p in zip(gold_b, pred_b_normal):
    cm[g, p] += 1
print(f"{'gold|pred':>16s}" + "".join(f"{c:>16s}" for c in LABELS_B) + f"{'total':>8s}")
for i, c in enumerate(LABELS_B):
    print(f"{c:>16s}" + "".join(f"{cm[i, j]:16d}" for j in range(3)) + f"{cm[i].sum():8d}")
print(f"{'total':>16s}" + "".join(f"{cm[:, j].sum():16d}" for j in range(3)) + f"{cm.sum():8d}")

never_pred = [LABELS_B[j] for j in range(3) if cm[:, j].sum() == 0]
print("\nLEITURA:")
if never_pred:
    print(f"  CONFIRMA: a cabeça B NUNCA prediz {never_pred} na val em condição normal.")
    print("  REFUTA o mecanismo declarado da H1 como causa do −0,076 se DIRECT estiver")
    print("  nessa lista: o delta só pode vir de movimento entre as classes restantes.")
    print("  Também explica o fmeasure_hard = NaN sem precisar de hipótese nova.")
else:
    print("  As 3 classes aparecem no argmax — o eixo DIRECT↔JUDGEMENTAL É alcançável,")
    print("  logo o mecanismo da H1 permanece na mesa. O NaN do fmeasure vem de outro")
    print("  motivo (interno ao PyEvALL), não de classe nunca predita.")
print(f"\n  (referência: eval_proxy_task_b_f1 = {FINAL['eval_proxy_task_b_f1']:.4f}, "
      f"eval_task_b_fmeasure_hard = {FINAL['eval_task_b_fmeasure_hard']})")



### 3.2 TESTE 2 — ruído pareado, 5 derangements, e permutação within-class

**Decide H3** (hipótese nula: a cabeça B é frágil a *qualquer* perturbação do vetor
fundido, e o −0,076 não mede informação de vídeo).

Quatro arms, todos sobre o slot de vídeo, com ‖Δ‖ **reportado** para cada um:

| arm | o que é | serve para |
|---|---|---|
| `perm_across` | derangement livre (5 seeds) | a sonda original |
| `noise_across` | gaussiano isotrópico com ‖ε‖ = ‖Δ‖ de `perm_across` | H3 |
| `perm_within` | derangement **dentro da mesma classe gold de B** | H1 vs H3 |
| `noise_within` | gaussiano com ‖ε‖ = ‖Δ‖ de `perm_within` | controle de magnitude |

**Por que o arm within-class existe** (não estava na §5 do doc 15): ruído isotrópico só é
decisivo numa direção. Se ruído ≥ permutação → H3 confirmada, limpo. Se ruído <
permutação, é **ambíguo**: "permutação é in-distribution-mas-errada" é uma explicação,
"a cabeça é mais robusta a direções OOD que a direções plausíveis" é outra, e nenhuma das
duas fala de alinhamento vídeo↔rótulo. O arm within-class fala: sob H3, within e across
machucam igual; sob H1, within machuca **menos**, porque o vetor trocado ainda carrega a
assinatura de formato da classe certa.

**Por que `noise_within` existe:** ‖Δ‖ de within-class deve ser menor que de across, e
"machuca menos" não vale nada se estiver confundido com "foi perturbado menos". O par
`perm_within` vs `noise_within` compara dano a magnitude casada.

**Regra pré-registada** (repetida da célula de abertura): só há diferença entre arms se
ela exceder a amplitude dos 5 derangements.


In [ ]:

rng = np.random.RandomState(20260809)
A_V, B_V = SLOTS["video"]
vid_base = FUSION_IN[:, A_V:B_V].float()

# ---- gold hard de B para o arm within-class (sentinela = grupo próprio) ----
group = np.full(N_VAL, -1, dtype=int)
group[mask_b] = gold_b
groups_desc = {int(g): int((group == g).sum()) for g in sorted(set(group.tolist()))}
print("grupos para permutação within-class (gold hard de B; -1 = sem voto válido):")
for g, n in groups_desc.items():
    name = "SEM_VOTO" if g == -1 else LABELS_B[g]
    print(f"  {name:>16s}  n={n}")


def derangement_within(seed):
    """Derangement dentro de cada grupo. Grupos de tamanho 1 ficam fixos (contados)."""
    rs = np.random.RandomState(1000 + seed)
    idx = np.arange(N_VAL)
    fixed = 0
    for g in set(group.tolist()):
        pos = np.where(group == g)[0]
        if len(pos) < 2:
            fixed += len(pos)
            continue
        for _ in range(500):
            p = rs.permutation(len(pos))
            if not (p == np.arange(len(pos))).any():
                break
        idx[pos] = pos[p]
        fixed += int((idx[pos] == pos).sum())
    return idx, fixed


def gaussian_at(target_norm, seed):
    """Ruído gaussiano com ‖ε‖₂ = target_norm em TODA amostra (média casada por construção)."""
    g = torch.from_numpy(np.random.RandomState(2000 + seed).randn(N_VAL, B_V - A_V).astype("float32"))
    g = g / g.norm(dim=1, keepdim=True) * target_norm
    return vid_base + g


arms = {}          # nome -> lista de (metrics, delta)
# perm_across
for s in DERANGEMENT_SEEDS:
    p = derangement(N_VAL, s)
    arms.setdefault("perm_across", []).append(
        (metrics_of(replay(perturb("video", index=p))), delta_norm("video", index=p)))
# perm_within
fixed_total = None
for s in DERANGEMENT_SEEDS:
    p, fixed = derangement_within(s)
    fixed_total = fixed
    arms.setdefault("perm_within", []).append(
        (metrics_of(replay(perturb("video", index=p))), delta_norm("video", index=p)))

d_across = float(np.mean([d for _, d in arms["perm_across"]]))
d_within = float(np.mean([d for _, d in arms["perm_within"]]))
print(f"\namostras sem par no grupo (ficaram fixas no arm within): {fixed_total}")
print(f"‖Δvid_feat‖ médio — across={d_across:.3f}  within={d_within:.3f}")

# noise arms, casados em magnitude
for s in DERANGEMENT_SEEDS:
    r = gaussian_at(d_across, s)
    arms.setdefault("noise_across", []).append(
        (metrics_of(replay(perturb("video", replacement=r))), delta_norm("video", replacement=r)))
    r = gaussian_at(d_within, 100 + s)
    arms.setdefault("noise_within", []).append(
        (metrics_of(replay(perturb("video", replacement=r))), delta_norm("video", replacement=r)))


def summarize(arm_name, key):
    vals = np.array([m[key] for m, _ in arms[arm_name]])
    base = m_normal[key]
    return vals.mean() - base, vals.max() - vals.min(), vals.mean()


print(f"\n{'arm':>14s}{'‖Δ‖':>9s}" + "".join(f"{k.replace('task_','').replace('icmnorm_','').replace('icmsoftnorm_',''):>16s}" for k in METRIC_KEYS))
print(f"{'normal':>14s}{0.0:9.3f}" + "".join(f"{m_normal[k]:16.4f}" for k in METRIC_KEYS))
for arm in ("perm_across", "noise_across", "perm_within", "noise_within"):
    dn = float(np.mean([d for _, d in arms[arm]]))
    row = f"{arm:>14s}{dn:9.3f}"
    for k in METRIC_KEYS:
        dmean, amp, _ = summarize(arm, k)
        row += f"{dmean:+11.4f}±{amp:.3f}"
    print(row)
print("\n(cada célula: Δ médio vs normal ± amplitude sobre os 5 derangements/seeds)")

KB = "task_b_icmnorm_hard"
dp, ap, _ = summarize("perm_across", KB)
dn_, an, _ = summarize("noise_across", KB)
dw, aw, _ = summarize("perm_within", KB)
dnw, anw, _ = summarize("noise_within", KB)
tol = max(ap, an, aw, anw)

print("\nLEITURA (Task B ICMNorm hard) — arvore CORRIGIDA em 2026-08-09.")
print("A versao anterior desta celula so comparava noise_across vs perm_across e")
print("retornava antes de consultar os arms within. Rotulava 'indistinguivel' como")
print("'CONFIRMA H3' e ignorava os dois arms que existiam justamente para discriminar.")
print("Nao e violacao de pre-registro: a regra estava incompleta e assimetrica.\n")
for arm, (dm, amp) in (("perm_across", (dp, ap)), ("noise_across", (dn_, an)),
                       ("perm_within", (dw, aw)), ("noise_within", (dnw, anw))):
    print(f"  {arm:14s} D={dm:+.4f} +-{amp:.4f}   ||Dvid_feat||="
          f"{float(np.mean([d for _, d in arms[arm]])):.3f}")
print(f"  tolerancia pre-registada = {tol:.4f}\n")

# --- discriminador PRIMARIO: within vs across, a magnitude praticamente casada ---
print(f"  [1] perm_within vs perm_across: |{dw:+.4f} - {dp:+.4f}| = {abs(dw - dp):.4f} "
      f"{'>' if abs(dw - dp) > tol else '<='} tol")
if abs(dw - dp) > tol and dw > dp:
    print("      REFUTA H3 na forma forte. As duas permutacoes tem ||D|| praticamente")
    print("      igual e a unica coisa que difere entre elas e a CLASSE GOLD DE B DO")
    print("      DOADOR — que o modelo nunca ve. Uma cabeca genericamente fragil nao")
    print("      pode dar dano ~zero sob uma perturbacao dessa magnitude.")
    print("      CONFIRMA o mecanismo da H1: o que o slot de video carrega e informacao")
    print("      de classe de B (registro/formato da fonte).")
    if abs(dw) < 0.01:
        print("      E MAIS: dano ~zero dentro da classe significa que o slot de video NAO")
        print("      carrega nada ALEM da classe de B. Se carregasse intensidade, categoria")
        print("      de C ou detalhe de enquadramento, trocar dentro da classe custaria")
        print("      algo. O ramo de video funciona como um classificador fraco de")
        print("      intencao da fonte, e nada mais. E uma afirmacao de TETO.")
elif abs(dw - dp) <= tol:
    print("      INCONCLUSIVO: trocar dentro da classe machuca o mesmo que entre classes.")
    print("      Compativel com H3 (fragilidade generica) — o slot nao carrega classe de B.")
else:
    print("      ANOMALIA nao prevista: within machuca MAIS que across. Reportar, nao ler.")

# --- controle de magnitude ---
print(f"\n  [2] perm_within vs noise_within (||D|| casada): |{dw:+.4f} - {dnw:+.4f}| = "
      f"{abs(dw - dnw):.4f} {'>' if abs(dw - dnw) > tol else '<='} tol")
if abs(dw - dnw) > tol:
    print("      Corrobora [1] a magnitude casada: ruido de mesma norma machuca, troca")
    print("      dentro da classe nao. O dano depende do CONTEUDO da perturbacao.")
else:
    print("      Abaixo da tolerancia — mesma direcao de [1], mas nao sustenta sozinho.")

# --- o teste original, agora no seu lugar: baixo poder ---
print(f"\n  [3] perm_across vs noise_across (o teste da §5 do doc 15): "
      f"|{dp:+.4f} - {dn_:+.4f}| = {abs(dp - dn_):.4f} "
      f"{'>' if abs(dp - dn_) > tol else '<='} tol")
if abs(dp - dn_) <= tol:
    print("      SEM PODER a n=505. Isso NAO confirma H3 — 'indistinguivel' nao e")
    print("      'confirmado'. Nota metodologica para a tese: ruido isotropico era o")
    print("      controle ERRADO; quem discrimina e permutacao condicional a classe com")
    print("      magnitude casada.")
else:
    print("      Separa: a permutacao machuca mais que ruido de mesma magnitude.")

print(f"\n  CAVEAT do desenho: perm_within agrupa pelo GOLD de B, logo o arm e oracular")
print(f"  na CONSTRUCAO. E legitimo — o gold escolhe o doador, nunca entra no modelo —")
print(f"  mas tem de ser declarado antes que alguem levante.")
print(f"\n  CORRECAO DE NUMERO: com 5 derangements, perm_across da {dp:+.4f} +-{ap:.4f}.")
print(f"  O -0,076 do doc 15 (2 seeds) era um ponto fragil: a amplitude e da ordem do")
print(f"  proprio efeito. Reportar SEMPRE com a amplitude, nunca como valor pontual.")
print("\n  Nenhum destes numeros e evidencia de que o video AJUDA (video_core.py:1069).")



## 4. TESTE 1 — escada de permutação por modalidade

**Decide H1.** Mesma sonda, mesmas 5 seeds de derangement, aplicada a cada um dos quatro
slots. Os arms ficam comparáveis porque compartilham a permutação (mesma seed → mesmo
índice), como pede a §5 do doc 15.

O que se lê: a **quota de dependência** por modalidade na mesma escala. Se permutar texto
derrubar a Task A dezenas de pontos e vídeo derrubar 0,009, o resultado do vídeo deixa de
ser anômalo — vira proporcional, e o achado é sobre o dataset, não sobre um bug.


In [ ]:

ladder = {"video": arms["perm_across"]}     # já calculado no teste 2 — não recomputar
for slot in ("audio", "texto", "bio"):
    ladder[slot] = []
    for s in DERANGEMENT_SEEDS:
        p = derangement(N_VAL, s)
        ladder[slot].append((metrics_of(replay(perturb(slot, index=p))),
                             delta_norm(slot, index=p)))

print(f"{'slot':>8s}{'dim':>6s}{'‖Δ‖':>9s}" +
      "".join(f"{k.replace('task_','').replace('icmnorm_','H_').replace('icmsoftnorm_','S_'):>15s}" for k in METRIC_KEYS))
print(f"{'normal':>8s}{'':>6s}{0.0:9.3f}" + "".join(f"{m_normal[k]:15.4f}" for k in METRIC_KEYS))
for slot in ("video", "audio", "texto", "bio"):
    a, b = SLOTS[slot]
    dn = float(np.mean([d for _, d in ladder[slot]]))
    row = f"{slot:>8s}{b - a:6d}{dn:9.3f}"
    for k in METRIC_KEYS:
        vals = np.array([m[k] for m, _ in ladder[slot]])
        row += f"{vals.mean() - m_normal[k]:+10.4f}±{vals.max() - vals.min():.3f}"
    print(row)
print("\n(Δ médio vs normal ± amplitude sobre 5 derangements; ‖Δ‖ = perturbação aplicada ao slot)")

print("\nLEITURA:")
for k in ("task_a_icmnorm_hard", "task_b_icmnorm_hard", "task_c_icmnorm_hard"):
    deltas = {s: float(np.mean([m[k] for m, _ in ladder[s]])) - m_normal[k]
              for s in ("video", "audio", "texto", "bio")}
    order = sorted(deltas, key=lambda s: deltas[s])
    print(f"  {k:26s} hierarquia (mais nocivo → menos): " +
          " > ".join(f"{s}({deltas[s]:+.3f})" for s in order))
dv = float(np.mean([m["task_a_icmnorm_hard"] for m, _ in ladder["video"]])) - m_normal["task_a_icmnorm_hard"]
dt = float(np.mean([m["task_a_icmnorm_hard"] for m, _ in ladder["texto"]])) - m_normal["task_a_icmnorm_hard"]
print()
if dt < dv - 0.05:
    print("  CONFIRMA H1 (parte 1): o texto domina a Task A por larga margem. A")
    print("  contribuição pequena do vídeo em A é PROPORCIONAL, não anômala — o conteúdo")
    print("  sexista chega pela transcrição.")
else:
    print("  REFUTA H1 (parte 1): permutar texto NÃO domina a Task A. O modelo não está a")
    print("  ler conteúdo de nenhuma modalidade com força — o alvo passa a ser a fusão /")
    print("  as cabeças, não o encoder de vídeo.")
print("  ⚠ Estes Δ medem DEPENDÊNCIA do modelo treinado, não teto de informação da")
print("     modalidade. Um ramo que o treino aprendeu a ignorar dá Δ≈0 mesmo que a")
print("     modalidade contenha sinal.")



## 5. TESTE 3 — diagnóstico das features e do pooling

**Decide H2, H5 e fecha o caveat da §3.4 do doc 15.**

### 5.1 Escalas por ramo e contribuição efetiva na fusão

A §3.4 do doc 15 afirma que "a fusão não ignora o vídeo" com base na norma média das
colunas de `fusion.0.weight` (vídeo 0,749 > texto 0,630). O próprio doc declara o caveat:
**contribuição = peso × escala da feature**, e as escalas nunca foram medidas. Aqui são.

Reporto duas coisas:
- ‖feature‖₂ média por amostra, por slot;
- ‖W[:, slot] @ feat_slot‖₂ média por amostra — a contribuição do slot ao pré-ativação da
  fusão.

⚠ A segunda é **descritiva, não causal**: há bias e GELU a jusante, então o pré-ativação
não se decompõe linearmente na saída. A quantidade causal é o que a escada de permutação
mede.


In [ ]:

W = model.fusion[0].weight.detach().float().cpu()        # [1024, 2688]
b_fus = model.fusion[0].bias.detach().float().cpu()

print(f"{'slot':>8s}{'dim':>6s}{'‖feat‖ méd':>13s}{'‖feat‖ med':>13s}"
      f"{'‖col W‖ méd':>14s}{'‖W·feat‖ méd':>15s}{'% da soma':>11s}")
contribs = {}
for slot in ("video", "audio", "texto", "bio"):
    a, b = SLOTS[slot]
    f = FUSION_IN[:, a:b].float()
    nf = f.norm(dim=1)
    colw = W[:, a:b].norm(dim=0)
    contrib = (f @ W[:, a:b].T).norm(dim=1)
    contribs[slot] = float(contrib.mean())
tot = sum(contribs.values())
for slot in ("video", "audio", "texto", "bio"):
    a, b = SLOTS[slot]
    f = FUSION_IN[:, a:b].float()
    nf = f.norm(dim=1)
    colw = W[:, a:b].norm(dim=0)
    print(f"{slot:>8s}{b - a:6d}{nf.mean():13.3f}{nf.median():13.3f}"
          f"{colw.mean():14.4f}{contribs[slot]:15.3f}{100 * contribs[slot] / tot:10.1f}%")
print(f"{'bias':>8s}{'':>6s}{'':>13s}{'':>13s}{'':>14s}{b_fus.norm():15.3f}")

print("\nLEITURA (fecha o caveat da §3.4 do doc 15):")
cv, ct = contribs["video"], contribs["texto"]
if cv >= ct:
    print("  CONFIRMA §3.4: a contribuição efetiva do slot de vídeo ao pré-ativação da")
    print("  fusão é ≥ a do texto. A hipótese 'a fusão aprendeu a descartar o vídeo' morre")
    print("  também no eixo peso×escala, não só no eixo peso.")
elif cv < 0.5 * ct:
    print("  CORRIGE §3.4: a norma de coluna maior do vídeo é enganosa. Em peso×escala o")
    print("  vídeo contribui menos da metade do texto — a leitura 'a fusão dá ao vídeo")
    print("  peso nominal maior que ao texto' NÃO deve ir para a tese sem esta ressalva.")
else:
    print("  §3.4 sobrevive parcialmente: vídeo abaixo do texto em peso×escala, mas na")
    print("  mesma ordem de grandeza. Nem 'ignorado' nem 'dominante'.")
print("  ⚠ Descritivo, não causal (bias + GELU a jusante).")



### 5.2 O `vid_feat` varia entre amostras?

Se o encoder devolvesse quase o mesmo vetor para todo vídeo, o slot seria um bias
constante e a permutação não poderia machucar nada. Métrica: ‖std entre amostras‖ /
‖vetor médio‖, e a distribuição de similaridade cosseno entre pares.


In [ ]:

print(f"{'slot':>8s}{'‖média‖':>11s}{'‖std‖':>11s}{'std/média':>12s}"
      f"{'cos médio':>12s}{'cos p05':>10s}{'cos p95':>10s}")
for slot in ("video", "audio", "texto", "bio"):
    a, b = SLOTS[slot]
    f = FUSION_IN[:, a:b].float()
    mu, sd = f.mean(dim=0), f.std(dim=0)
    fn = f / f.norm(dim=1, keepdim=True).clamp_min(1e-9)
    sub = fn[np.random.RandomState(0).choice(N_VAL, 300, replace=False)]
    cos = (sub @ sub.T)
    iu = torch.triu_indices(300, 300, offset=1)
    cv_ = cos[iu[0], iu[1]]
    print(f"{slot:>8s}{mu.norm():11.3f}{sd.norm():11.3f}{sd.norm() / mu.norm():12.3f}"
          f"{cv_.mean():12.3f}{cv_.quantile(0.05):10.3f}{cv_.quantile(0.95):10.3f}")

a, b = SLOTS["video"]
fv = FUSION_IN[:, a:b].float()
ratio = float(fv.std(dim=0).norm() / fv.mean(dim=0).norm())
print("\nLEITURA:")
if ratio < 0.15:
    print(f"  CONFIRMA a forma forte da H2: std/média = {ratio:.3f} — o vid_feat é quase")
    print("  constante entre amostras. O slot de vídeo funciona como bias, e o encoder")
    print("  praticamente não distingue os vídeos deste dataset.")
else:
    print(f"  REFUTA a forma forte da H2: std/média = {ratio:.3f} — o vid_feat varia entre")
    print("  amostras. O encoder distingue os vídeos; a questão é se o que distingue é")
    print("  semântica de sexismo ou propriedade de captura (→ teste 4).")



### 5.3 O pooling: entropia do softmax e perfil temporal

**Correção de arquitetura que sai desta investigação independentemente do resto.**
`TemporalAttentionPool.forward` (`video_core.py:964-967`) faz `softmax(logits, dim=1)`
sobre `x = [B, 1568, 768]`, que é o `last_hidden_state` do VideoMAE
(`video_core.py:1128-1129`). Os 1 568 tokens são **espaço-temporais** (8 posições
temporais × 196 patches 14×14), não frames.

**Isto é o que o código faz, não hipótese**, e não depende de nenhuma suposição sobre
ordenação de tokens: `dim=1` é o eixo dos 1 568, ponto. A docstring atual ("pooling sobre
a dimensão temporal", "cada token de frame recebe um peso escalar") está errada nas duas
afirmações, e a descrição de arquitetura da tese precisa ser corrigida.

**Separado disso** (e mais fraco): o reshape `1568 → (8, 196)` para ler o perfil temporal
assume que a ordem de flatten do `Conv3d` de patches é `(t, h, w)` com `t` mais externo.
`[SUPOSIÇÃO]` — afeta **só** o diagnóstico de perfil temporal, nunca a correção acima.
A célula seguinte faz uma checagem empírica dessa suposição.

Selectividade reportada como **tokens efetivos = exp(H)** ao lado dos nats.
`ln(1568) = 7,357` ⇒ uniforme = 1 568 tokens efetivos (pool ≡ média).


In [ ]:

ps = cache["pool_stats"]                      # [505, 3] = (entropia nats, max w, top10)
ent, wmax, top10 = ps[:, 0], ps[:, 1], ps[:, 2]
eff = ent.exp()
H_UNIF = math.log(1568)

print(f"entropia uniforme ln(1568) = {H_UNIF:.3f} nats  ->  1568 tokens efetivos\n")
print(f"{'estatística':>22s}{'média':>10s}{'mediana':>10s}{'p05':>10s}{'p95':>10s}")
for name, v in (("entropia (nats)", ent), ("tokens efetivos exp(H)", eff),
                ("peso máximo", wmax), ("massa do top-10", top10)):
    print(f"{name:>22s}{v.mean():10.3f}{v.median():10.3f}"
          f"{v.quantile(0.05):10.3f}{v.quantile(0.95):10.3f}")

tprof = cache["pool_temporal"]                # [505, 8], soma dos pesos por posição temporal
print(f"\nperfil temporal médio (massa de atenção por posição temporal, [SUPOSIÇÃO] reshape):")
print("  " + "  ".join(f"t{i}={tprof[:, i].mean():.3f}" for i in range(N_TEMPORAL)))
print(f"  desvio-padrão ENTRE amostras do perfil: {tprof.std(dim=0).mean():.4f}")
print(f"  (uniforme seria {1 / N_TEMPORAL:.3f} em cada posição)")

print("\nLEITURA:")
if eff.median() > 800:
    print(f"  CONFIRMA colapso para MÉDIA: {eff.median():.0f} tokens efetivos de 1568 — o")
    print("  pool é praticamente `mean(dim=1)`. Os 98 561 params do agregador não")
    print("  compraram seletividade, e o 'attention pooling' da tese é, na prática, média")
    print("  global. Note que isto NÃO explica o padrão B≫A≈C sozinho (afetaria as três).")
elif eff.median() < 20:
    print(f"  CONFIRMA colapso para POUCOS TOKENS: {eff.median():.0f} tokens efetivos — o")
    print("  'vídeo' que chega à fusão é um punhado de patches espaço-temporais.")
else:
    print(f"  REFUTA as duas formas de colapso: {eff.median():.0f} tokens efetivos,")
    print("  seletividade intermediária. O pool escolhe, e H5 na forma forte não se")
    print("  sustenta como gargalo primário.")
print("  CORREÇÃO PARA A TESE (independente de tudo acima): o softmax roda sobre os 1568")
print("  tokens espaço-temporais, não sobre posições temporais. O pool NÃO é temporal.")



### 5.4 Checagem empírica do reshape `1568 → (8, 196)`

Substituo os frames 0–1 de uma amostra (o **primeiro tubelet**) pelos de outra e vejo em
qual dos 8 blocos de 196 tokens o `last_hidden_state` muda mais.

⚠ **Limite declarado:** com self-attention completa, *todo* token muda quando qualquer
entrada muda. O teste mostra **concentração**, não prova. Se o bloco 0 dominar, a ordem
`(t, h, w)` está sustentada; se a mudança for uniforme, o perfil temporal da célula
anterior fica sem base e deve ser omitido da tese.


In [ ]:

it0, it1 = val_ds[0], val_ds[1]
f0 = it0["frames"].unsqueeze(0).to(device)
f_mix = f0.clone()
f_mix[0, 0:2] = it1["frames"][0:2].to(device)

with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
    h_ref = model.videomae(pixel_values=f0).last_hidden_state.float()
    h_mix = model.videomae(pixel_values=f_mix).last_hidden_state.float()
assert h_ref.shape[1] == 1568, f"esperado 1568 tokens, veio {h_ref.shape[1]}"

d = (h_mix - h_ref).norm(dim=-1).squeeze(0).view(N_TEMPORAL, N_SPATIAL).mean(dim=1).cpu()
print("‖Δhidden‖ média por bloco de 196 tokens (frames 0-1 trocados):")
for i, v in enumerate(d):
    bar = "#" * int(40 * v / d.max())
    print(f"  bloco {i}: {v:8.4f}  {bar}")
print(f"\nrazão bloco0 / média dos outros = {d[0] / d[1:].mean():.2f}×")
print("\nLEITURA:")
if d.argmax().item() == 0 and (d[0] / d[1:].mean()) > 1.5:
    print("  SUSTENTA a suposição de reshape: a troca do primeiro tubelet concentra-se no")
    print("  bloco 0. A ordem de flatten é (t, h, w) com t mais externo, e o perfil")
    print("  temporal da célula 5.3 pode ser lido.")
else:
    print("  NÃO SUSTENTA: a mudança não se concentra no bloco 0. O perfil temporal da")
    print("  célula 5.3 fica SEM BASE — omitir da tese. A correção sobre o softmax rodar")
    print("  em 1568 tokens NÃO é afetada (não depende de ordenação).")



## 6. TESTE 4 — sondas lineares sobre os caches

**Decide o que `vid_feat` realmente codifica.** Regressão logística sobre features
**congeladas** para prever: rótulo A, rótulo B, língua do vídeo, duração (mediana). A
mesma sonda sobre `txt_feat` e `aud_feat` serve de referência.

**Anti-leakage:** ajuste numa metade da val, report na outra, usando `vc.split_val_halves`
— as **mesmas** metades para todos os alvos e todas as features.

**Nulo por permutação** (adição não prevista na §5, mas obrigatória): com n≈252 no report
e d=768, uma sonda decora. Sem o nulo, "duração é previsível" pode ser artefato de
dimensionalidade. Cada alvo leva 10 repetições com rótulos embaralhados; o número que se
lê é **AUC − AUC_nulo**.

⚠ Diferenças de AUC abaixo de ~0,05 não são interpretáveis a este n.


In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

IDX_FIT, IDX_REP = vc.split_val_halves(N_VAL)
print(f"metades (vc.split_val_halves): fit={len(IDX_FIT)}  report={len(IDX_REP)}")

# ---- alvos ----
dur = pd.read_csv("analysis/audio_durations.csv")
dur = dur[dur["split"] == "training"].copy()
dur["id_EXIST"] = dur["id_EXIST"].astype(str)
dmap = dict(zip(dur["id_EXIST"], dur["duration_s"]))
dur_v = np.array([dmap.get(i, np.nan) for i in VAL_IDS], dtype=float)
assert np.isfinite(dur_v).sum() == N_VAL, (
    f"join de duração incompleto: {int(np.isfinite(dur_v).sum())} de {N_VAL}. "
    "Verificar dtype de id_EXIST e o filtro split=='training'.")

lang_v = val_df["lang"].values
targets = {
    "A_hard (sexista?)": (LAB_A > 0.5).astype(int),
    "B_hard (3 classes)": group,                                    # -1 = sem voto
    "língua (es vs en)": (lang_v == "es").astype(int),
    "duração > mediana": (dur_v > np.median(dur_v)).astype(int),
}
print("\ndistribuição dos alvos:")
for name, y in targets.items():
    u, c = np.unique(y, return_counts=True)
    print(f"  {name:22s} {dict(zip(u.tolist(), c.tolist()))}")


def probe(X, y, seed_shuffle=None):
    """AUC (macro-ovr se multiclasse) na metade de report. Fit só na metade de fit."""
    m = y >= 0 if y.min() < 0 else np.ones(len(y), bool)
    if seed_shuffle is not None:
        y = y.copy()
        y[m] = np.random.RandomState(seed_shuffle).permutation(y[m])
    fit = np.array([i for i in IDX_FIT if m[i]])
    rep = np.array([i for i in IDX_REP if m[i]])
    if len(np.unique(y[fit])) < 2 or len(np.unique(y[rep])) < 2:
        return float("nan")
    sc = StandardScaler().fit(X[fit])
    clf = LogisticRegression(max_iter=3000, C=0.05).fit(sc.transform(X[fit]), y[fit])
    p = clf.predict_proba(sc.transform(X[rep]))
    if p.shape[1] == 2:
        return float(roc_auc_score(y[rep], p[:, 1]))
    return float(roc_auc_score(y[rep], p, multi_class="ovr", average="macro",
                              labels=clf.classes_))


feats = {s: FUSION_IN[:, a:b].float().numpy() for s, (a, b) in SLOTS.items()}
print(f"\n{'alvo':>22s}" + "".join(f"{s:>26s}" for s in ("video", "texto", "audio", "bio")))
print(f"{'':>22s}" + "".join(f"{'AUC (AUC−nulo)':>26s}" for _ in range(4)))
probe_out = {}
for name, y in targets.items():
    row = f"{name:>22s}"
    for s in ("video", "texto", "audio", "bio"):
        auc = probe(feats[s], y)
        null = np.array([probe(feats[s], y, seed_shuffle=1000 + r) for r in range(10)])
        gain = auc - np.nanmean(null)
        probe_out[(name, s)] = (auc, float(np.nanmean(null)), gain)
        row += f"{auc:16.3f} ({gain:+.3f})"
    print(row)

print("\nLEITURA (sonda sobre video):")
gA = probe_out[("A_hard (sexista?)", "video")][2]
gB = probe_out[("B_hard (3 classes)", "video")][2]
gL = probe_out[("língua (es vs en)", "video")][2]
gD = probe_out[("duração > mediana", "video")][2]
sem = max(gA, gB)
capt = max(gL, gD)
print(f"  ganho sobre o nulo — semântica: A {gA:+.3f} / B {gB:+.3f} | "
      f"captura: língua {gL:+.3f} / duração {gD:+.3f}")
if capt > sem + 0.05:
    print("  CONFIRMA H2: vid_feat prediz propriedades de CAPTURA (língua/duração) melhor")
    print("  que os rótulos. O encoder codifica aparência e estatística global, não")
    print("  semântica social. Se duração é previsível, o confound temporal da F12 (taxa")
    print("  de amostragem função da duração) VAZA para o modelo — análogo visual do")
    print("  ARCH-04, e isso é achado novo.")
elif sem > capt + 0.05:
    print("  REFUTA H2 na forma forte: a informação de rótulo ESTÁ em vid_feat e domina a")
    print("  de captura. Se assim for, o gargalo é a jusante (fusão/cabeças), não o")
    print("  encoder — aponta para H5/H3, não para trocar de encoder.")
else:
    print("  INCONCLUSIVO a n≈{}: diferenças abaixo de 0,05 não são interpretáveis."
          .format(len(IDX_REP)))
print("  ⚠ Uma sonda linear boa NÃO prova que o modelo usa essa informação; e uma sonda")
print("     ruim não prova que a informação não está lá em forma não-linear.")



## 7. TESTE 5 — matriz de flips da Task B sob permutação

**Decide H1.** Para onde vão as predições de B quando o slot de vídeo é permutado: se os
flips se concentram no eixo DIRECT↔JUDGEMENTAL, o visual entrega registro/formato — o
mecanismo exato que a H1 propõe. Flips difusos e proporcionais às marginais apontam para
H3.

Agregado sobre os 5 derangements. Os casos que mudaram vão para um CSV com a transcrição,
para leitura manual (é assim que se verifica a sub-hipótese "a transcrição de um duet
crítico contém a fala sexista citada").


In [ ]:

flip = np.zeros((3, 3), dtype=int)     # pred normal -> pred permutado
flip_rows = []
texts = val_df["text"].values
for s in DERANGEMENT_SEEDS:
    p = derangement(N_VAL, s)
    lb = replay(perturb("video", index=p))[1]
    pn = cache["logits_b"].argmax(axis=1)
    pp = lb.argmax(axis=1)
    for i in range(N_VAL):
        if not mask_b[i]:
            continue
        flip[pn[i], pp[i]] += 1
        if pn[i] != pp[i]:
            flip_rows.append({
                "seed": s, "id_EXIST": VAL_IDS[i], "lang": lang_v[i],
                "gold_b": LABELS_B[int(LAB_B[i].argmax())],
                "pred_normal": LABELS_B[int(pn[i])], "pred_perm": LABELS_B[int(pp[i])],
                "donor_id": VAL_IDS[int(p[i])],
                "donor_gold_b": (LABELS_B[int(LAB_B[p[i]].argmax())]
                                 if mask_b[p[i]] else "SEM_VOTO"),
                "transcript": str(texts[i])[:400],
            })

n_eval = int(mask_b.sum()) * len(DERANGEMENT_SEEDS)
print(f"agregado sobre {len(DERANGEMENT_SEEDS)} derangements × {int(mask_b.sum())} amostras "
      f"= {n_eval} decisões")
print(f"flips: {len(flip_rows)} ({100 * len(flip_rows) / n_eval:.1f} %)\n")
print(f"{'normal|perm':>16s}" + "".join(f"{c:>16s}" for c in LABELS_B) + f"{'total':>8s}")
for i, c in enumerate(LABELS_B):
    print(f"{c:>16s}" + "".join(f"{flip[i, j]:16d}" for j in range(3)) + f"{flip[i].sum():8d}")

off = flip.copy()
np.fill_diagonal(off, 0)
if off.sum() > 0:
    dj = off[LABELS_B.index("DIRECT"), LABELS_B.index("JUDGEMENTAL")] + \
         off[LABELS_B.index("JUDGEMENTAL"), LABELS_B.index("DIRECT")]
    print(f"\nflips no eixo DIRECT<->JUDGEMENTAL: {dj} de {off.sum()} "
          f"({100 * dj / off.sum():.1f} %)")
    for i, c in enumerate(LABELS_B):
        for j, c2 in enumerate(LABELS_B):
            if i != j and off[i, j]:
                print(f"  {c} -> {c2}: {off[i, j]} ({100 * off[i, j] / off.sum():.1f} %)")

out_csv = CACHE_DIR / "task_b_flips.csv"
pd.DataFrame(flip_rows).to_csv(out_csv, index=False)
print(f"\ncasos que mudaram gravados em {out_csv} (com transcrição, para leitura manual)")

print("\nLEITURA:")
if off.sum() == 0:
    print("  Nenhum flip: a permutação do vídeo não muda NENHUMA decisão hard de B. Todo o")
    print("  Δ de ICM-Norm hard vem de reordenação de probabilidades, não de decisões —")
    print("  releitura importante do −0,076.")
elif dj / off.sum() > 0.5:
    print("  CONFIRMA o mecanismo da H1: mais da metade dos flips está no eixo")
    print("  DIRECT<->JUDGEMENTAL. O visual entrega registro/formato, que é exatamente o")
    print("  que a taxonomia da Task B discrimina.")
else:
    print("  NÃO CONFIRMA a H1: os flips são difusos / não se concentram no eixo")
    print("  DIRECT<->JUDGEMENTAL. Compatível com H3 (fragilidade genérica da cabeça).")
print("  Passo manual pendente: ler o CSV e verificar se os JUDGEMENTAL cuja transcrição")
print("  'parece DIRECT' são os que mudam — só o visual desambigua esses.")



## 8. TESTE 6 — o resumo dos 1 568 tokens joga sinal fora?

**Não estava na §5 do doc 15.** Entra porque o §5.3 mediu que o pool não agrega,
**seleciona**: peso mediano do token máximo = **0,502**, massa mediana do top-10 =
**0,970**. E o §5.4 mediu que os tokens são **locais** (troca de tubelet concentra
10,88× no bloco correspondente), logo o token eleito é um *recorte* do vídeo, não um
resumo dele.

Pergunta que isto abre e que se responde **sem treinar**: a informação de rótulo está
nos outros ~1 563 tokens que o seletor descarta, ou não está em token nenhum?

### As quatro variantes de resumo

| variante | dim | o que isola |
|---|---:|---|
| `pool_learned` | 768 | o `vid_feat` atual — baseline |
| `mean_all` | 768 | média simples dos 1 568 (o que o pool substituiu) |
| `mean_temporal` | 6 144 | média espacial dentro de cada uma das 8 fatias, concatenadas — preserva o eixo do tempo |
| `top5` | 768 | só os 5 tokens eleitos, renormalizados — testa se a cauda 97 %→100 % faz algo |

### Decisões de design

- **Só o ramo de vídeo roda.** Preciso do `last_hidden_state`, não do XLM-R.
  `model.videomae` + `model.video_pool.attn` corta o forward em ~2/3.
- **Os tokens não são guardados.** 505 × 1 568 × 768 × 4 B = 2,4 GB. Guardo as 4
  variantes: ~17 MB.
- **Portão desta seção:** `pool_learned` recomputado tem de casar com o slot de vídeo já
  cacheado (`fusion_in[:, 0:768]`). Se não casar, o forward não reproduz e nada aqui vale.
- **O nulo por permutação é obrigatório aqui**, não opcional: `mean_temporal` tem 8× mais
  features que as outras, e é o nulo que absorve a vantagem de dimensionalidade. Comparar
  AUC cru entre variantes de dimensão diferente seria enganoso.
- **Escala de ruído:** além das metades fixas de `vc.split_val_halves` (comparáveis,
  anti-leakage, e que reproduzem os números do §6), 8 divisões 50/50 aleatórias dão média
  ± desvio do AUC. Sem barra de erro, comparar 0,069 com 0,11 não significa nada.

### Leitura pré-registada, deliberadamente ASSIMÉTRICA

- `mean_all` **ganhar** de `pool_learned` por mais que a tolerância → há sinal na cauda
  que o seletor descarta → **gatilho** para um braço treinado (cross-attention sobre os
  1 568 tokens, ou pooling fatorado). Não é prova.
- `mean_all` **empatar ou perder**, ambos fracos → **nenhum agregador salva**: a
  informação não está nos tokens. Cross-attention seria uma forma mais cara de não achar
  nada. Este é o resultado que FECHA a opção, e é achado negativo publicável.
- Sonda linear mede decodificabilidade de um vetor **congelado**, não o que uma fusão
  treinada faria. A assimetria é a mesma regra de `video_core.py:1066-1071`.


In [ ]:

import contextlib

from torch.utils.data import DataLoader

VARIANTS_FILE = CACHE_DIR / "pool_variants.pt"
N_TOP = 5


def _compute_variants(out_path, disable_lora=False):
    """Forward SO do ramo de video, devolvendo as 4 variantes de resumo.

    `disable_lora=True` roda dentro de `model.videomae.disable_adapter()`: ZERA o delta
    LoRA e deixa os pesos base. Os q_bias/v_bias restaurados pelo ARCH-01 sao params BASE
    congelados (nao LoRA) e permanecem — e o que torna a comparacao limpa.
    """
    dl = DataLoader(val_ds, batch_size=2, shuffle=False,
                    collate_fn=vc.video_collate, num_workers=2)   # 0 se der problema
    acc = {k: [] for k in ("pool_learned", "mean_all", "mean_temporal", "top5")}
    lora_ctx = (model.videomae.disable_adapter() if disable_lora
                else contextlib.nullcontext())
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16), lora_ctx:
        for bi, batch in enumerate(dl):
            h = model.videomae(pixel_values=batch["frames"].to(device)).last_hidden_state
            B = h.shape[0]
            assert h.shape[1] == 1568, h.shape
            lg = model.video_pool.attn(h).squeeze(-1)                  # [B, 1568]
            w = torch.softmax(lg, dim=1)
            acc["pool_learned"].append((w.unsqueeze(-1) * h).sum(1).float().cpu())
            acc["mean_all"].append(h.mean(1).float().cpu())
            acc["mean_temporal"].append(
                h.view(B, N_TEMPORAL, N_SPATIAL, vc.VIDEO_DIM).mean(2)
                 .reshape(B, N_TEMPORAL * vc.VIDEO_DIM).float().cpu())
            vals, idx = w.topk(N_TOP, dim=1)                          # [B, 5]
            tok = torch.gather(h, 1, idx.unsqueeze(-1).expand(-1, -1, vc.VIDEO_DIM))
            wn = vals / vals.sum(1, keepdim=True)
            acc["top5"].append((wn.unsqueeze(-1) * tok).sum(1).float().cpu())
            if bi % 50 == 0:
                print(f"  batch {bi}/{len(dl)}", flush=True)
    out = {k: torch.cat(v, dim=0) for k, v in acc.items()}
    torch.save(out, out_path)
    return out


if VARIANTS_FILE.exists():
    VAR = torch.load(VARIANTS_FILE, weights_only=False)
    print(f"[cache] variantes carregadas de {VARIANTS_FILE} (forward saltado)")
else:
    print("[forward] rodando SO o ramo de video sobre a val...")
    VAR = _compute_variants(VARIANTS_FILE)
    print(f"[cache] gravado em {VARIANTS_FILE}")

for k, v in VAR.items():
    print(f"  {k:14s} {tuple(v.shape)}")
    assert v.shape[0] == N_VAL

# ---- PORTAO: pool_learned recomputado == slot de video do cache do Stage A ----
_a, _b = SLOTS["video"]
ref_slot = FUSION_IN[:, _a:_b].float()
got = VAR["pool_learned"].float()
cos = torch.nn.functional.cosine_similarity(got, ref_slot, dim=1)
mx = float((got - ref_slot).abs().max())
print("\nPORTAO da secao 8: cosseno(pool_learned recomputado, slot cacheado)")
print(f"  media={cos.mean():.6f}  min={cos.min():.6f}  maxD absoluto={mx:.4f}")
assert float(cos.mean()) > 0.9995 and float(cos.min()) > 0.995, (
    "PORTAO DA SECAO 8 FALHOU: o pool recomputado NAO reproduz o slot de video do "
    "Stage A. O forward so-video divergiu do forward completo (modo train? dtype?). "
    "Nada desta secao pode ser lido.")
print(">>> PORTAO OK: o forward so-video reproduz o ramo de video do forward completo.")


In [ ]:

NULL_REPS, SPLIT_REPS = 10, 8


def probe_split(X, y, fit, rep, seed_shuffle=None):
    """AUC na metade `rep`, ajustado so na metade `fit`. Mesma config da secao 6."""
    m = y >= 0 if y.min() < 0 else np.ones(len(y), bool)
    if seed_shuffle is not None:
        y = y.copy()
        y[m] = np.random.RandomState(seed_shuffle).permutation(y[m])
    f = np.array([i for i in fit if m[i]])
    r = np.array([i for i in rep if m[i]])
    if len(np.unique(y[f])) < 2 or len(np.unique(y[r])) < 2:
        return float("nan")
    sc = StandardScaler().fit(X[f])
    clf = LogisticRegression(max_iter=1000, C=0.05).fit(sc.transform(X[f]), y[f])
    pr = clf.predict_proba(sc.transform(X[r]))
    if pr.shape[1] == 2:
        return float(roc_auc_score(y[r], pr[:, 1]))
    return float(roc_auc_score(y[r], pr, multi_class="ovr", average="macro",
                               labels=clf.classes_))


RAND_SPLITS = []
for _r in range(SPLIT_REPS):
    _perm = np.random.RandomState(7000 + _r).permutation(N_VAL)
    RAND_SPLITS.append((_perm[:N_VAL // 2], _perm[N_VAL // 2:]))

VNAMES = ["pool_learned", "mean_all", "mean_temporal", "top5"]
Xs = {k: VAR[k].float().numpy() for k in VNAMES}
gains, sds, aucs = {}, {}, {}
print("rodando sondas (10-20 min de CPU; mean_temporal tem 6144 features)...")
for t, y in targets.items():
    for v in VNAMES:
        auc = probe_split(Xs[v], y, IDX_FIT, IDX_REP)
        null = np.array([probe_split(Xs[v], y, IDX_FIT, IDX_REP, seed_shuffle=1000 + k)
                         for k in range(NULL_REPS)])
        rand = np.array([probe_split(Xs[v], y, f, r) for f, r in RAND_SPLITS])
        aucs[(v, t)] = auc
        gains[(v, t)] = auc - float(np.nanmean(null))
        sds[(v, t)] = float(np.nanstd(rand))
    print(f"  ok: {t}", flush=True)

print(f"\n{'alvo':>22s}" + "".join(f"{v:>22s}" for v in VNAMES))
print(f"{'':>22s}" + "".join(f"{'AUC-nulo (+-sd)':>22s}" for _ in VNAMES))
for t in targets:
    print(f"{t:>22s}" + "".join(
        f"{gains[(v, t)]:+14.3f} (+-{sds[(v, t)]:.3f})" for v in VNAMES))

print("\ncross-check com a secao 6 (mesma feature, mesmas metades):")
for t in targets:
    ref = probe_out[(t, "video")][2]
    g = gains[("pool_learned", t)]
    print(f"  {t:>22s} sec6={ref:+.3f}  sec8={g:+.3f}  D={g - ref:+.4f}")

print("\nLEITURA (pre-registada):")
for t in ("A_hard (sexista?)", "B_hard (3 classes)"):
    gp, gm = gains[("pool_learned", t)], gains[("mean_all", t)]
    gt, g5 = gains[("mean_temporal", t)], gains[("top5", t)]
    thr = max(0.05, 2 * max(sds[("pool_learned", t)], sds[("mean_all", t)]))
    print(f"\n  {t}   (tolerancia = {thr:.3f} = max(0,05 ; 2sd))")
    print(f"    pool_learned {gp:+.3f} | mean_all {gm:+.3f} | "
          f"mean_temporal {gt:+.3f} | top5 {g5:+.3f}")
    if gm > gp + thr:
        print("    GATILHO: a media simples de TODOS os 1568 tokens decodifica o rotulo")
        print("    MELHOR que o seletor aprendido. Ha sinal na cauda que o pool descarta.")
        print("    Justifica UM braco treinado com agregador que preserve os tokens")
        print("    (cross-attention sobre os 1568 — criterio de reabertura de")
        print("    video_core.py:1061-1064, agora atendido). NAO e prova.")
    elif abs(gm - gp) <= thr:
        print("    EMPATE: media simples e seletor decodificam igual. O seletor nao esta")
        print("    perdendo sinal LINEARMENTE decodificavel, e trocar o agregador nao tem")
        print("    alvo obvio. Combinado com o nivel absoluto baixo, empurra para 'a")
        print("    informacao nao esta nos tokens', que FECHA a opcao (achado negativo).")
    else:
        print("    O seletor GANHA da media simples: ele faz trabalho real. Trocar por")
        print("    mean puro seria regressao. Fecha a opcao 'mean puro'.")
    if gt > max(gp, gm) + thr:
        print("    E MAIS: preservar o eixo TEMPORAL (mean_temporal) ganha das duas ->")
        print("    a estrutura temporal carrega sinal que qualquer pooling global destroi.")
        print("    Esse e o braco a treinar, nao a cross-attention generica.")
    if abs(g5 - gp) <= thr:
        print("    top5 ~= pool_learned: a cauda de 1563 tokens nao adiciona nada ao")
        print("    vetor atual — confirma que o pool e, de facto, um seletor de 5 tokens.")

print("\n  ASSIMETRIA (video_core.py:1066-1071): 'mean_all ganha' e GATILHO para gastar")
print("  uma run; 'mean_all nao ganha' FECHA a opcao. Sonda mede decodificabilidade")
print("  linear de vetor congelado, nunca o que uma fusao treinada faria.")



## 8b. TESTE 7 — quanto a adaptação LoRA comprou? (tokens COM vs SEM LoRA)

**Última sonda antes de gastar uma run.** Todas as sondas anteriores mediram tokens
produzidos pelo LoRA atual. Nenhuma diz o que tokens **diferentes** conteriam.

Contexto medido: ‖(α/r)·B·A‖_F ≈ 0,80 contra ‖W‖_F ≈ 38 da matriz base — a atualização é
**~2 %** em norma, confinada a rank ≤ 16 de 768. O encoder está perto do regime de sonda
linear. Esta célula mede o **retorno da adaptação**: quanto de decodificabilidade os 589 824
params de LoRA compraram sobre o VideoMAE puro.

### O que "LoRA off" significa aqui

`model.videomae` é um `PeftModel`; `disable_adapter()` zera o ΔW. Os `q_bias`/`v_bias`
restaurados pelo ARCH-01 são params **base congelados**, não LoRA — permanecem. Mesmo
checkpoint base, só a adaptação removida.

### Decisão de design que decide o teste: qual pooling usar

O `pool_learned` foi treinado **sobre tokens adaptados**. Aplicá-lo a tokens base
confundiria "os tokens mudaram" com "o pool está descasado com a entrada". Então a
comparação honesta usa só resumos **sem parâmetros treinados**:

- `mean_all` (adaptado) vs `mean_all` (base)
- `mean_temporal` (adaptado) vs `mean_temporal` (base)

Operação idêntica nas duas condições. `pool_learned` também é reportado, **rotulado como
confundido**, apenas para completude.

### Portão: no-op silencioso

Se `disable_adapter()` não tiver efeito, tudo parecerá "o LoRA não faz diferença" — que é
indistinguível da conclusão verdadeira e levaria à decisão errada. A célula mede a distância
entre tokens base e adaptados e **aborta** se forem iguais.

### Leitura pré-registada

- `adaptado ≈ base` nos rótulos → o LoRA a r=16 **não moveu** a decodificabilidade.
  Ambíguo entre "falta capacidade" (→ braço r=64/MLP só no vídeo, 1 run) e "o pré-treino de
  reconstrução de pixels não tem o que oferecer" (→ trocar de encoder). Restringe, não
  resolve.
- `adaptado ≫ base` → a adaptação **funciona** e escalá-la é bem motivado; o braço de mais
  capacidade deixa de ser aposta.
- `adaptado < base` → a adaptação **piorou** a decodificabilidade linear. Compatível com o
  LoRA ter otimizado a loss por uma via que a sonda não vê; reportar como anomalia.
- **Alvo de nuisance como diagnóstico:** se o LoRA subiu `duração`/`língua` mais que os
  rótulos, os 2 % de ΔW foram gastos a ajustar propriedade de captura.


In [ ]:

BASE_FILE = CACHE_DIR / "pool_variants_base.pt"

print("tipo do encoder de video:", type(model.videomae).__name__)
assert hasattr(model.videomae, "disable_adapter"), (
    "model.videomae nao expoe disable_adapter() — nao e um PeftModel nesta versao do "
    "peft. Sem isso este teste nao pode rodar.")

if BASE_FILE.exists():
    VAR_BASE = torch.load(BASE_FILE, weights_only=False)
    print(f"[cache] variantes SEM LoRA carregadas de {BASE_FILE} (forward saltado)")
else:
    print("[forward] rodando o ramo de video com o adaptador LoRA DESLIGADO...")
    VAR_BASE = _compute_variants(BASE_FILE, disable_lora=True)
    print(f"[cache] gravado em {BASE_FILE}")

# ---- PORTAO: disable_adapter() teve efeito? ----
print("\nPORTAO: distancia entre tokens BASE e ADAPTADOS (resumo mean_all)")
_ad = VAR["mean_all"].float()
_bs = VAR_BASE["mean_all"].float()
_cos = torch.nn.functional.cosine_similarity(_ad, _bs, dim=1)
_rel = (_ad - _bs).norm(dim=1) / _ad.norm(dim=1).clamp_min(1e-9)
print(f"  cosseno(adaptado, base): media={_cos.mean():.6f}  min={_cos.min():.6f}")
print(f"  ||adaptado - base|| / ||adaptado||: media={_rel.mean():.4f}  "
      f"p05={_rel.quantile(0.05):.4f}  p95={_rel.quantile(0.95):.4f}")
assert float(_rel.mean()) > 1e-3, (
    "PORTAO FALHOU: os tokens com e sem LoRA sao praticamente IDENTICOS "
    f"(desvio relativo medio {float(_rel.mean()):.2e}). disable_adapter() foi no-op — "
    "provavelmente o adaptador nao esta ativo no caminho medido. Um no-op faria tudo "
    "parecer 'o LoRA nao faz diferenca', que e indistinguivel da conclusao verdadeira. "
    "Nada desta secao pode ser lido.")
print(">>> PORTAO OK: desligar o adaptador muda os tokens de forma mensuravel.")
print(f"    (referencia: ||DW||_F/||W||_F ~ 2 % nos pesos; aqui o efeito nos TOKENS "
      f"e {100 * float(_rel.mean()):.2f} %)")


In [ ]:

PARAM_FREE = ["mean_all", "mean_temporal"]
CONFOUNDED = ["pool_learned"]
Xs_base = {k: VAR_BASE[k].float().numpy() for k in PARAM_FREE + CONFOUNDED}

gains_base, sds_base = {}, {}
print("rodando sondas sobre tokens SEM LoRA (10-15 min de CPU)...")
for t, y in targets.items():
    for v in PARAM_FREE + CONFOUNDED:
        auc = probe_split(Xs_base[v], y, IDX_FIT, IDX_REP)
        null = np.array([probe_split(Xs_base[v], y, IDX_FIT, IDX_REP, seed_shuffle=1000 + k)
                         for k in range(NULL_REPS)])
        rand = np.array([probe_split(Xs_base[v], y, f, r) for f, r in RAND_SPLITS])
        gains_base[(v, t)] = auc - float(np.nanmean(null))
        sds_base[(v, t)] = float(np.nanstd(rand))
    print(f"  ok: {t}", flush=True)

print("\nAUC - nulo, por condicao. RESUMOS SEM PARAMETROS (comparacao valida):")
for v in PARAM_FREE:
    print(f"\n  --- {v} ---")
    print(f"  {'alvo':>22s}{'COM LoRA':>16s}{'SEM LoRA':>16s}{'ganho da adaptacao':>21s}")
    for t in targets:
        ga, gb = gains[(v, t)], gains_base[(v, t)]
        sd = max(sds[(v, t)], sds_base[(v, t)])
        thr = max(0.05, 2 * sd)
        mark = "  <-- move" if abs(ga - gb) > thr else ""
        print(f"  {t:>22s}{ga:+11.3f} (+-{sds[(v, t)]:.3f}){gb:+11.3f} "
              f"(+-{sds_base[(v, t)]:.3f}){ga - gb:+13.3f}{mark}")

print("\n  --- pool_learned (CONFUNDIDO: o pool foi treinado sobre tokens ADAPTADOS; "
      "\n      aplicado a tokens base mede tambem o descasamento) ---")
for t in targets:
    print(f"  {t:>22s}  COM={gains[('pool_learned', t)]:+.3f}  "
          f"SEM={gains_base[('pool_learned', t)]:+.3f}  "
          f"D={gains[('pool_learned', t)] - gains_base[('pool_learned', t)]:+.3f}")

print("\nLEITURA (pre-registada) — so os resumos SEM parametros contam:")
for t in ("A_hard (sexista?)", "B_hard (3 classes)"):
    print(f"\n  {t}")
    moved = []
    for v in PARAM_FREE:
        ga, gb = gains[(v, t)], gains_base[(v, t)]
        thr = max(0.05, 2 * max(sds[(v, t)], sds_base[(v, t)]))
        d = ga - gb
        moved.append(d > thr)
        print(f"    {v:14s} COM {ga:+.3f} | SEM {gb:+.3f} | ganho {d:+.3f} "
              f"(tolerancia {thr:.3f})")
    if any(moved):
        print("    A ADAPTACAO FUNCIONA: o LoRA moveu a decodificabilidade do rotulo acima")
        print("    da tolerancia. Escalar capacidade (r=64 ou target_modules com MLP, so no")
        print("    video) deixa de ser aposta e passa a ser extrapolacao de um efeito medido.")
        print("    Braco indicado: 1 run, E congelado pela calibracao, so o video muda.")
    else:
        print("    A ADAPTACAO NAO MOVEU nada mensuravel. Com ||DW|| = 2 % da matriz base,")
        print("    os tokens sao essencialmente features de PRE-TREINO do VideoMAE.")
        print("    Isso NAO decide entre as duas causas — 'falta capacidade' e 'o objetivo")
        print("    de reconstrucao de pixels nao tem o que oferecer' predizem o mesmo aqui.")
        print("    Mas muda o custo esperado: o braco de mais capacidade passa a ser aposta")
        print("    contra um retorno medido de ZERO no primeiro incremento de adaptacao.")

print("\n  DIAGNOSTICO DE NUISANCE — no que os 2 % de DW foram gastos:")
for v in PARAM_FREE:
    d_lab = max(gains[(v, "A_hard (sexista?)")] - gains_base[(v, "A_hard (sexista?)")],
                gains[(v, "B_hard (3 classes)")] - gains_base[(v, "B_hard (3 classes)")])
    d_nui = max(gains[(v, "duração > mediana")] - gains_base[(v, "duração > mediana")],
                gains[(v, "língua (es vs en)")] - gains_base[(v, "língua (es vs en)")])
    print(f"    {v:14s} ganho maximo em ROTULO {d_lab:+.3f} | "
          f"em NUISANCE (duracao/lingua) {d_nui:+.3f}")
    if d_nui > d_lab + 0.05:
        print("      A adaptacao subiu NUISANCE mais que rotulo — os params de LoRA foram")
        print("      gastos a codificar propriedade de captura. Achado novo, reportavel.")

print("\n  ⚠ Sonda linear sobre vetores congelados, n≈252 na metade de report. Nao mede o")
print("    que uma fusao treinada extrairia de forma nao-linear. A regra assimetrica de")
print("    video_core.py:1066-1071 continua: nada aqui e evidencia de que o video ajuda.")



## 9. Resumo — o que estes números decidem

A célula abaixo não computa nada novo: junta os veredictos numa tabela única, para colar
no documento 15 como seção de resultados.

**O que sai desta investigação independentemente de qualquer resultado numérico:**

1. **Correção de arquitetura para a tese.** O `TemporalAttentionPool` não é temporal: o
   softmax roda sobre os 1 568 tokens espaço-temporais (`video_core.py:964-967` ×
   `1128-1129`). Espaço e tempo competem no mesmo softmax e o peso é um escalar por token,
   logo nenhuma informação de *onde na tela* ou *quando no vídeo* sobrevive à soma. A
   docstring atual está errada nas duas afirmações. Isso é o que o código faz.
2. **A lição metodológica das sondas.** Zerar uma entrada **normalizada** não apaga a
   modalidade: zero pós-normalização é o pixel médio do ImageNet, uma imagem cinza
   plausível, que o encoder converte num embedding constante absorvido como bias — e as
   métricas até *melhoram*. Zerar `vid_feat` pós-pooling subestima por construção, porque
   reproduz exatamente a operação do modality dropout, para a qual o treino criou
   robustez. Permutação preserva a distribuição de entrada e mede o alinhamento. Isso é
   material publicável por si só.


In [ ]:

print("=" * 78)
print("RESUMO — sondas sobre", RUN_DIR, "| val =", N_VAL, "| sem treino, sem re-extração")
print("=" * 78)

print("\n[0] PORTÃO: replay fusion→heads reproduz o forward completo e as permutações")
print("    já medidas em test.ipynb. Todos os números abaixo são exatos, não aproximados.")

print("\n[3.1] Task B — classes efetivamente preditas em condição normal:",
      [LABELS_B[j] for j in range(3) if cm[:, j].sum() > 0])

print("\n[2] H3 (Task B ICMNorm hard, Δ vs normal, 5 seeds):")
for arm in ("perm_across", "noise_across", "perm_within", "noise_within"):
    dm, amp, _ = summarize(arm, KB)
    dn = float(np.mean([d for _, d in arms[arm]]))
    print(f"      {arm:14s} Δ={dm:+.4f} ±{amp:.4f}   ‖Δvid_feat‖={dn:.3f}")
print(f"      tolerância pré-registada = {tol:.4f}")

print("\n[1] Escada de permutação (Δ médio, 5 seeds):")
print(f"      {'slot':>8s}" + "".join(f"{k.replace('task_','').replace('icmnorm_','H_').replace('icmsoftnorm_','S_'):>13s}" for k in METRIC_KEYS))
for slot in ("video", "audio", "texto", "bio"):
    print(f"      {slot:>8s}" + "".join(
        f"{float(np.mean([m[k] for m, _ in ladder[slot]])) - m_normal[k]:+13.4f}"
        for k in METRIC_KEYS))

print("\n[3] Escalas e contribuição na fusão (peso × escala, descritivo):")
for slot in ("video", "audio", "texto", "bio"):
    a, b = SLOTS[slot]
    print(f"      {slot:>8s} ‖feat‖={FUSION_IN[:, a:b].float().norm(dim=1).mean():8.3f}  "
          f"‖W·feat‖={contribs[slot]:8.3f} ({100 * contribs[slot] / tot:4.1f} %)")
print(f"      pool: {eff.median():.0f} tokens efetivos de 1568 (uniforme = 1568)")
print(f"      vid_feat std/média entre amostras = {ratio:.3f}")

print("\n[4] Sondas lineares (AUC − nulo, metade de report):")
for name in targets:
    print(f"      {name:>22s}" + "".join(
        f"  {s}={probe_out[(name, s)][2]:+.3f}" for s in ("video", "texto", "audio", "bio")))

print("\n[5] Flips de B sob permutação:", len(flip_rows), "de", n_eval, "decisões;",
      f"eixo DIRECT<->JUDGEMENTAL = {dj if off.sum() else 0} de {off.sum()}")

print("\n[6] Comparacao de resumos dos 1568 tokens (AUC - nulo):")
for _t in ("A_hard (sexista?)", "B_hard (3 classes)"):
    print(f"      {_t:>22s}" + "".join(
        f"  {_v}={gains[(_v, _t)]:+.3f}" for _v in VNAMES))
print(f"      pool: peso do token maximo (mediana) = {float(wmax.median()):.3f}  |  "
      f"massa top-10 = {float(top10.median()):.3f}")

print("\n[7] Retorno da adaptacao LoRA (resumos SEM parametros, AUC - nulo):")
for _v in PARAM_FREE:
    for _t in ("A_hard (sexista?)", "B_hard (3 classes)", "duração > mediana"):
        print(f"      {_v:14s} {_t:>22s}  COM={gains[(_v, _t)]:+.3f}  "
              f"SEM={gains_base[(_v, _t)]:+.3f}  ganho={gains[(_v, _t)] - gains_base[(_v, _t)]:+.3f}")
print(f"      efeito de desligar o LoRA nos tokens: {100 * float(_rel.mean()):.2f} % "
      f"da norma de mean_all")

print("\n" + "=" * 78)
print("REGRA QUE NÃO MUDA (video_core.py:1069): nada aqui é evidência de que o vídeo")
print("ajuda. As sondas são GATILHO para gastar runs num braço treinado. Quem prova é")
print("o braço treinado.")
print("=" * 78)

# ── fecha o log (IDEMPOTENTE: re-executar esta celula nao levanta) ─────────
sys.stdout.flush()
if isinstance(sys.stdout, _Tee):
    _fh = sys.stdout.f
    sys.stdout = sys.stdout.term
    _fh.close()
    print(f"\n[log] relatorio completo gravado em {LOG_PATH} "
          f"({Path(LOG_PATH).stat().st_size} bytes)")
else:
    print(f"\n[log] tee ja estava fechado (celula re-executada). O log de {LOG_PATH} "
          f"e da PRIMEIRA execucao; para regravar, rodar a celula 0 de novo.")
